In [ ]:
include("../src/PhasorNetworks.jl")
using .PhasorNetworks

In [ ]:
using Plots, DifferentialEquations

In [ ]:
n_x = 101
n_y = 101
n_vsa = 1

In [ ]:
repeats = 6
tspan = (0.0, repeats*1.0)

In [ ]:
phases = collect([[x, y] for x in range(-1.0, 1.0, n_x), y in range(-1.0, 1.0, n_y)]) |> stack
phases = reshape(phases, (1,2,:));

In [ ]:
b = v_bind(phases, dims=2); 
ub = v_unbind(phases[1:1,1:1,:], phases[1:1,2:2,:]);

In [ ]:
spk_args = SpikingArgs(t_window=0.01, solver = Tsit5(), threshold=0.001)

In [ ]:
st_x = phase_to_train(phases[1:1,1:1,:], spk_args, repeats = repeats)
st_y = phase_to_train(phases[1:1,2:2,:], spk_args, repeats = repeats)

In [ ]:
soln = v_bind(st_x, st_y, spk_args = spk_args, tspan = tspan, return_solution=true)

In [ ]:
tbase = collect(0.0:0.01:6.0);

In [ ]:
p_out = solution_to_phase(soln, tbase, offset=0.0, spk_args=spk_args);

In [ ]:
p_out |> size

In [ ]:
using Statistics: mean

In [ ]:
plot(p_out[1,1,3768, :])

In [ ]:
scatter(p_out[1,1,:,end], vec(b))

In [ ]:
phase_error = vec(p_out[1,1,:,end]) .- vec(b)
mean_error = phase_error |> mean

In [ ]:
mean_error

In [ ]:
using .PhasorNetworks: functional_solution_to_potential

In [ ]:
u_out = functional_solution_to_potential(soln, tbase);

In [ ]:
u_out |> size

In [ ]:
function cplot(x, tbase)
    plot(tbase, real.(x))
    plot!(tbase, imag.(x))
end

In [ ]:
cplot(u_out[1,1,3768,:], tbase)

In [ ]:
cplot(u_out[1,1,5000,:], tbase)

In [ ]:
spk = v_bind(st_x, st_y, tspan = tspan, return_solution=false, spk_args = spk_args)

In [ ]:
pp = train_to_phase(spk, spk_args);

In [ ]:
pp |> size

In [ ]:
pp_err = pp[end-1, :, :, :] .- b;

In [ ]:
function remove_nan(x)
    y = filter(x -> !isnan(x), x)
    return y
end

In [ ]:
mean(remove_nan(pp_err))

In [ ]:
histogram(sin.(pi.*(remove_nan(pp_err))))